# NYC Bike Data: Dataset Description & Exploratory Analysis

This notebook documents and explores every dataset used in the project and underpins the
**technical report** in `docs/technical-report/`. It covers, in order of importance:

1. **Citi Bike dataset**: the project's main data source with bike trip records (one monthly file, downloaded manually).
2. **Weather**: hourly NYC observations from the Open-Meteo archive (fetched live).
3. **Bike lanes**: NYC bike-route network from NYC OpenData (fetched live).
4. **Station metadata**: current Citi Bike stations from the Lyft GBFS feed (fetched live).

For each dataset we describe its **structure**, run **exploratory data analysis (EDA)**, and
assess **data quality**. See `src/notebooks/README.md` for how to obtain the ride data and run
this notebook.

## 0. Setup

In [ ]:
import json
import re
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import folium
from folium.plugins import HeatMap

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

# Where to find the manually-downloaded Citi Bike monthly CSV(s) (relative to this notebook).
DATA_DIR = Path("data")

# Data sources (mirrors src/ingestion/config.yaml and src/backend/config.yaml).
WEATHER_API_URL = "https://archive-api.open-meteo.com/v1/archive"
BIKE_ROUTES_URL = "https://data.cityofnewyork.us/api/views/mzxg-pwib/rows.csv?accessType=DOWNLOAD"
GBFS_INFO_URL = "https://gbfs.lyft.com/gbfs/2.3/bkn/en/station_information.json"
GBFS_STATUS_URL = "https://gbfs.lyft.com/gbfs/2.3/bkn/en/station_status.json"
NYC_LAT, NYC_LON = 40.7823234, -73.9654161

# Distance-feature constants (see src/ingestion/sources/distances.py).
EARTH_RADIUS_KM = 6371
STREET_CIRCUITY_FACTOR = 1.3

Here we define an utility function for computing the Haversine distance calculated as:

$$d = 2 r \arcsin \sqrt{\sin^2\left(\frac{\Delta \phi}{2}\right) + \cos(\phi_1) \cos(\phi_2) \sin^2\left(\frac{\Delta \lambda}{2}\right)}$$

where $\phi$ is latitude, $\lambda$ is longitude, and $r$ is the Earth's radius (6371 km).

In [ ]:
# Vectorised great-circle distance (km) scaled by the street-circuity factor,
# matching the production formula in src/ingestion/sources/distances.py.
def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    return EARTH_RADIUS_KM * c * STREET_CIRCUITY_FACTOR

## 1. Citi Bike Trip Data

The core dataset Citi Bike publishes one CSV per month, each row a single trip, available from
the [system data page](https://www.citibikenyc.com/system-data). Download one month and place the
extracted CSV in the `data/` folder next to this notebook (see `README.md`).

Newer files (2020+) and the legacy (pre-2020) schema use different column names. This map normalises the legacy columns to the modern schema so the rest of the notebook works regardless of which month was downloaded.

In [ ]:
LEGACY_RENAME = {
    "starttime": "started_at",
    "stoptime": "ended_at",
    "start station name": "start_station_name",
    "start station id": "start_station_id",
    "end station name": "end_station_name",
    "end station id": "end_station_id",
    "start station latitude": "start_lat",
    "start station longitude": "start_lng",
    "end station latitude": "end_lat",
    "end station longitude": "end_lng",
}

def load_rides(data_dir: Path):
    csv_files = sorted(data_dir.glob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(
            f"No CSV files found in {data_dir.resolve()}. Download a Citi Bike monthly "
            "trip file, extract it, and place the CSV here. See src/notebooks/README.md."
        )
    df = pd.concat([pd.read_csv(f, low_memory=False) for f in csv_files], ignore_index=True)

    # Columns unique to the legacy schema — kept track of for the data-quality section.
    legacy_cols = [c for c in ["gender", "birth year", "bikeid", "tripduration", "usertype"]
                   if c in df.columns]
    is_legacy = "rideable_type" not in df.columns
    if is_legacy:
        df = df.rename(columns=LEGACY_RENAME)
        if "usertype" in df.columns:
            df["member_casual"] = df["usertype"].map(
                {"Subscriber": "member", "Customer": "casual"}
            ).fillna(df["usertype"])
        df["rideable_type"] = pd.NA          # absent before the e-bike rollout
        if "ride_id" not in df.columns:
            df["ride_id"] = df.index.astype(str)

    # Recent months carry fractional seconds meanwhile older months do not. Use a mixed format to handle both.
    for col in ["started_at", "ended_at"]:
        try:
            df[col] = pd.to_datetime(df[col], format="ISO8601")
        except (ValueError, TypeError):
            df[col] = pd.to_datetime(df[col], format="mixed")
    return df, {"files": [f.name for f in csv_files], "is_legacy": is_legacy,
                "legacy_cols": legacy_cols}

df, meta = load_rides(DATA_DIR)
print("Loaded files :", meta["files"])
print("Schema       :", "legacy (pre-2020)" if meta["is_legacy"] else "modern (2020+)")
print("Shape        :", df.shape)
df.head()

### 1.1 Structure

We provide a brief overview of the dataset's structure, including column names, data types, and sample records, through the `pandas` library. 

In [ ]:
df.info()

print("\nUnique rideable_type :", df["rideable_type"].dropna().unique().tolist())
print("Unique member_casual :", df["member_casual"].dropna().unique().tolist())

unique_stations = pd.concat([df["start_station_name"], df["end_station_name"]]).nunique()
print(f"Unique stations      : {unique_stations}")
print(f"Date range           : {df['started_at'].min()} -> {df['started_at'].max()}")

### 1.2 Trip Duration

Given the raw trip data, we compute a set of derived features, including trip duration and distance. This is done in order to enhance the analysis of user behaviour and trip patterns. 

In [ ]:
df["ride_length_s"] = (df["ended_at"] - df["started_at"]).dt.total_seconds()
df["ride_length_min"] = df["ride_length_s"] / 60

print("Ride length (minutes) summary:")
print((df["ride_length_s"] / 60).agg(["mean", "median", "min", "max", "std"]).round(2))

negative = (df["ride_length_s"] < 0).sum()
long_rides = (df["ride_length_s"] > 4 * 3600).sum()
print(f"\nRides with negative duration : {negative}")
print(f"Rides longer than 4 hours    : {long_rides}")

# Every trip is kept
plt.figure(figsize=(8, 5))
sns.histplot(df["ride_length_min"].clip(lower=0, upper=60), bins=60)
plt.axvline(df["ride_length_min"].median(), color="red", ls="--",
            label=f"median {df['ride_length_min'].median():.1f} min")
plt.title("Ride length (0\u201360 min)")
plt.xlabel("Ride length (min)")
plt.legend()
plt.tight_layout()
plt.show()

Ride length is strongly **right-skewed**: most trips last only a few to ~20 minutes (the
histogram is zoomed to the first hour), with a thin tail stretching to much longer rides. No
trips are excluded here. The few **negative** durations (clock/data glitches where a trip ends
before it starts) and rides over **4 hours** (more likely a bike that was never docked correctly
than a genuine 4-hour trip) are kept and folded into the edge bins; both are quantified above and
examined in the data-quality section below.

### 1.3 Trip Distance

In [ ]:
# Straight-line distance between start and end stations, scaled by the circuity factor
df["trip_distance_km"] = haversine_km(
    df["start_lat"], df["start_lng"], df["end_lat"], df["end_lng"]
)

print("Trip distance (km) summary:")
print(df["trip_distance_km"].agg(["mean", "median", "min", "max", "std"]))

# Nothing is excluded
cap = 10  # km; distances above this collapse into the last bin
plt.figure(figsize=(8, 5))
sns.histplot(df["trip_distance_km"].clip(upper=cap), bins=60)
plt.xlabel("Trip distance (km)")
plt.title("Trip distance (all trips; tail >= 10 km folded)")
plt.tight_layout()
plt.show()

This is the **straight-line** distance between the start and end stations, scaled by a 1.3
circuity factor to approximate the real on-street path. No trips are excluded: **round trips**
that begin and end at the same station appear as the **0 km** bar (the straight-line metric
cannot capture their real path), and the long-distance tail is folded into the final bin
so these anomalies stay visible. Both must be considered when interpreting the distribution or
computing distance-based statistics (e.g., average distance per trip, average speed).

### 1.4 Bike Type and User Type

The objective of this analysis is to understand the relationship between bike type and user type. The first panel shows the **bike-type mix within each user type**. Meanwhile the second panel displays the **median ride length by user and bike type**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bike-type mix *within* each user type (share, not raw volume)
counts = df.groupby(["member_casual", "rideable_type"]).size().reset_index(name="rides")
counts["share"] = 100 * counts["rides"] / counts.groupby("member_casual")["rides"].transform("sum")
sns.barplot(data=counts, x="member_casual", y="share", hue="rideable_type", ax=axes[0])
axes[0].set(title="Bike-type mix by user type", xlabel="User type",
            ylabel="Share of the user's rides (%)")

# Median ride length by user and bike type
med = (df.groupby(["member_casual", "rideable_type"])["ride_length_min"]
         .median().reset_index())
sns.barplot(data=med, x="member_casual", y="ride_length_min", hue="rideable_type", ax=axes[1])
axes[1].set(title="Median ride length by user & bike type", xlabel="User type",
            ylabel="Median ride length (min)")

# Add headroom and pin each legend to the top-right so it never overlaps a bar.
for ax in axes:
    ax.set_ylim(0, ax.get_ylim()[1] * 1.3)
    ax.legend(title="rideable_type", loc="upper right", framealpha=0.9)

plt.tight_layout()
plt.show()

# Overall composition, for context.
print("Share of rides by user type (%):")
print((df["member_casual"].value_counts(normalize=True) * 100).round(1))
print("\nShare of rides by bike type (%):")
print((df["rideable_type"].value_counts(normalize=True) * 100).round(1))

Raw volume is dominated by **members** (commuter use), but that count alone is not very
informative. Splitting each dimension by the other is more revealing. **Bike-type mix** (left):
**casual** riders lean more heavily on **electric** bikes than members do. **Ride length**
(right): casual riders take **longer** trips than members on both bike types, while members ride a
steady, short duration regardless of bike type, consistent with frequent short-hop commuting versus
occasional, mostly-electric leisure trips.

### 1.5 Temporal Patterns

Our purpose here is to explore the temporal patterns of bike usage, including daily and weekly trends, through the ride counts. 

In [ ]:
df["start_hour"] = df["started_at"].dt.hour
df["day_of_week"] = df["started_at"].dt.day_name()
df["date"] = df["started_at"].dt.date
df["day_type"] = np.where(
    df["started_at"].dt.weekday < 5, "Weekday", "Weekend"
)

# Mean rides per hour, separating weekdays from weekends.
hourly = (
    df.groupby(["date", "start_hour", "day_type"]).size().reset_index(name="rides")
)
mean_rides = hourly.groupby(["start_hour", "day_type"])["rides"].mean().reset_index()

plt.figure(figsize=(12, 5))
sns.barplot(x="start_hour", y="rides", hue="day_type", data=mean_rides)
plt.xlabel("Hour of day")
plt.ylabel("Mean rides per day")
plt.title("Mean rides by start hour: weekday vs weekend")
plt.tight_layout()
plt.show()

Weekdays show the classic bimodal commuting pattern (peaks around 8 AM and 5-6 PM), while
weekends follow a single midday hump consistent with leisure use.

### 1.6 Spatial Patterns

We explore the spatial patterns of bike usage, using a spatial heatmap to visualize the distribution of bike trips across the city. This analysis helps identify areas with high demand for bike-sharing services.

In [ ]:
# Heatmap of trip origins, weighted by how many trips start at each station.
origins = (
    df.dropna(subset=["start_lat", "start_lng", "start_station_name"])
    .groupby("start_station_name")
    .agg(lat=("start_lat", "first"), lon=("start_lng", "first"), rides=("ride_id", "count"))
    .reset_index()
)
bike_map = folium.Map(location=[origins["lat"].mean(), origins["lon"].mean()], zoom_start=12)
HeatMap(origins[["lat", "lon", "rides"]].values.tolist()).add_to(bike_map)
# recreate the map and add a tighter heatmap to reduce the glow
bike_map = folium.Map(location=[origins["lat"].mean(), origins["lon"].mean()], zoom_start=12)
HeatMap(
    origins[["lat", "lon", "rides"]].values.tolist(),
    radius=8,
    blur=6,
    min_opacity=0.3,
    max_zoom=18
).add_to(bike_map)
bike_map

In [ ]:
# Station-to-station flow: how many trips run between each ordered station pair.
flow = (
    df.dropna(subset=["start_station_name", "end_station_name"])
    .groupby(["start_station_name", "end_station_name"]).size()
    .reset_index(name="trips")
)
print(f"Distinct station pairs with at least one trip: {len(flow):,}")
print("\nTop 10 station-to-station flows:")
print(flow.sort_values("trips", ascending=False).head(10).to_string(index=False))

Demand is highly concentrated: a small set of stations near transit hubs and the business
districts dominate both departures and arrivals, so the heatmap clusters around Midtown and Lower
Manhattan, and the busiest station-to-station flows tend to link these same hubs.

### 1.7 Data Quality

In [ ]:
# Missing values per column.
missing = df.isna().sum()
missing = (
    pd.DataFrame({"missing": missing, "pct": (missing / len(df) * 100).round(2)})
    .query("missing > 0")
    .sort_values("missing", ascending=False)
)
print("Columns with missing values:")
print(missing if not missing.empty else "  none")

# Duplicate trip identifiers.
print(f"\nDuplicate ride_id values: {df['ride_id'].duplicated().sum()}")

# Invalid coordinates (outside a generous NYC bounding box, or null).
bad_coords = (
    df["start_lat"].between(40.4, 41.0) & df["start_lng"].between(-74.3, -73.6)
).eq(False).sum()
print(f"Trips with start coords outside the NYC bounding box: {bad_coords}")

# Negative / zero-length trips already quantified in 1.2.
print(f"Negative-duration trips : {(df['ride_length_s'] < 0).sum()}")
print(f"Zero-duration trips      : {(df['ride_length_s'] == 0).sum()}")

**Schema drift.** The Citi Bike file format changed in 2020. The loader above detects the legacy
layout and normalises its column names; legacy files additionally carry `gender`, `birth year`
and `bikeid` columns (absent in modern files) and lack `rideable_type` (no e-bikes yet). The
production ingestion pipeline (`src/ingestion/sources/rides.py`) reconciles both layouts the same
way.

In [ ]:
if meta["is_legacy"]:
    print("Loaded a LEGACY-format month.")
    print("Legacy-only columns present:", meta["legacy_cols"])
else:
    print("Loaded a MODERN-format month (no legacy-only columns).")

# Two station identifiers exist (numeric station_id and the public short_name used in trip data);
# they are not interchangeable across feeds — see the GBFS section below.
print("\nExample start_station_id values:", df["start_station_id"].dropna().astype(str).unique()[:5])

## 2. Weather Data

Hourly NYC weather from the [Open-Meteo archive API](https://open-meteo.com/), used to relate
ridership to weather. We fetch the full calendar year covering the ride data so seasonality is
visible even when only one ride month is loaded.

In [ ]:
ride_year = df["started_at"].min().year
start_date = date(ride_year, 1, 1)
end_date = min(date(ride_year, 12, 31), date.today() - timedelta(days=1))

resp = requests.get(
    WEATHER_API_URL,
    params={
        "latitude": NYC_LAT,
        "longitude": NYC_LON,
        "start_date": start_date.isoformat(),
        "end_date": end_date.isoformat(),
        "hourly": "temperature_2m,precipitation,weather_code,wind_speed_10m",
        "timezone": "America/New_York",
        "wind_speed_unit": "kmh",
    },
    timeout=(5, 120),
)
resp.raise_for_status()

weather = pd.DataFrame(resp.json()["hourly"])
weather["datetime"] = pd.to_datetime(weather["time"])
weather = weather.drop(columns="time")
print(f"Fetched {len(weather):,} hourly rows: {start_date} -> {end_date}")
weather.head()

### 2.1 Structure and Summary

In [ ]:
weather.info()
print("\nSummary statistics:")
weather[["temperature_2m", "wind_speed_10m", "precipitation", "weather_code"]].describe()

### 2.2 Distributions and Seasonality

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(weather["temperature_2m"], bins=40, ax=axes[0]).set(title="Temperature (°C)")
sns.histplot(weather["wind_speed_10m"], bins=40, ax=axes[1]).set(title="Wind speed (km/h)")
sns.histplot(weather.loc[weather["precipitation"] > 0, "precipitation"], bins=40, ax=axes[2]).set(
    title="Precipitation > 0 (mm)")
plt.tight_layout()
plt.show()

weather["month"] = weather["datetime"].dt.month
monthly_temp = weather.groupby("month")["temperature_2m"].mean()
plt.figure(figsize=(12, 4))
sns.lineplot(x=monthly_temp.index, y=monthly_temp.values, marker="o")
plt.xticks(range(1, 13))
plt.xlabel("Month")
plt.ylabel("Mean temperature (°C)")
plt.title(f"Mean monthly temperature, {ride_year}")
plt.tight_layout()
plt.show()

Temperature follows the expected annual cycle (cold winters, warm summers), while precipitation is
**zero for most hours** with occasional spikes — a heavily right-skewed variable, which is why the
histogram only shows the non-zero values.

### 2.3 Ridership vs. Weather

In [ ]:
# Join hourly ride counts to the matching weather hour and look at rides vs. temperature.
df["hour_ts"] = df["started_at"].dt.floor("h")
rides_per_hour = df.groupby("hour_ts").size().reset_index(name="rides")
merged = rides_per_hour.merge(weather, left_on="hour_ts", right_on="datetime", how="inner")

merged["temp_bin"] = (merged["temperature_2m"] // 5 * 5).astype(int)
by_temp = merged.groupby("temp_bin")["rides"].mean().reset_index()

plt.figure(figsize=(12, 5))
sns.barplot(x="temp_bin", y="rides", data=by_temp)
plt.xlabel("Temperature bin (°C)")
plt.ylabel("Mean rides per hour")
plt.title("Mean hourly ridership by temperature")
plt.tight_layout()
plt.show()

Ridership rises with temperature up to a comfortable range and drops off in the cold — confirming
weather is a useful predictor of demand, which is the main reason this dataset is included.

### 2.4 Data Quality

In [ ]:
missing_w = weather.isna().sum()
print("Missing values per column:")
print(missing_w[missing_w > 0] if missing_w.any() else "  none")
print(f"\nHours expected vs. present: "
      f"{int((end_date - start_date).total_seconds() // 3600) + 24} vs. {len(weather)}")

## 3. Bike Lanes

NYC's bike-route network from [NYC OpenData](https://data.cityofnewyork.us/) (dataset `mzxg-pwib`),
used to render bike infrastructure. Each row is a route segment with a `the_geom` geometry, the
street, facility class, borough, status and installation/retirement dates.

In [ ]:
routes = pd.read_csv(BIKE_ROUTES_URL, low_memory=False)
print("Shape:", routes.shape)
print("Columns:", routes.columns.tolist())
routes.head()

### 3.1 Breakdowns

In [ ]:
BORO = {1: "Manhattan", 2: "Bronx", 3: "Brooklyn", 4: "Queens", 5: "Staten Island"}
routes["boro_name"] = routes["boro"].map(BORO)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
routes["boro_name"].value_counts().plot.bar(ax=axes[0], title="Segments by borough")
routes["facilitycl"].value_counts().plot.bar(ax=axes[1], title="Segments by facility class")
routes["status"].value_counts().plot.bar(ax=axes[2], title="Segments by status")
for ax in axes:
    ax.set_ylabel("Segments")
plt.tight_layout()
plt.show()

Segments concentrate in Manhattan and Brooklyn, and most are **current** rather than retired. The
facility class encodes the level of protection (e.g. protected paths vs. shared/sharrow lanes),
which matters for how the network is rendered in the visualization.

### 3.2 Installation Timeline

In [ ]:
routes["install_year"] = pd.to_datetime(
    routes["instdate"], errors="coerce"
).dt.year
by_year = routes["install_year"].value_counts().sort_index()

plt.figure(figsize=(12, 4))
sns.barplot(x=by_year.index.astype(int), y=by_year.values)
plt.xticks(rotation=45)
plt.xlabel("Installation year")
plt.ylabel("Segments installed")
plt.title("Bike-route segments installed per year")
plt.tight_layout()
plt.show()

The installation timeline tracks how NYC's bike network has expanded over the years (note that
`instdate` is missing for some legacy segments, so the earliest years are under-counted).

### 3.3 Map of the Network

In [ ]:
# Extract polylines ([[lat, lon], ...]) from a (MULTI)LINESTRING WKT string.
def parse_wkt_lines(wkt):
    if not isinstance(wkt, str):
        return []
    lines = []
    for group in re.findall(r"\(([^()]+)\)", wkt):
        pts = []
        for pair in group.split(","):
            parts = pair.split()
            if len(parts) >= 2:
                lon, lat = float(parts[0]), float(parts[1])  # WKT is lon-lat
                pts.append([lat, lon])
        if pts:
            lines.append(pts)
    return lines

# Draw a sample of segments to keep the map light.
sample = routes.dropna(subset=["the_geom"]).head(800)
routes_map = folium.Map(location=[NYC_LAT, NYC_LON], zoom_start=11)
for geom in sample["the_geom"]:
    for line in parse_wkt_lines(geom):
        folium.PolyLine(line, color="crimson", weight=2, opacity=0.6).add_to(routes_map)
routes_map

### 3.4 Data Quality

In [ ]:
quality_cols = ["the_geom", "instdate", "ret_date", "facilitycl", "boro", "street"]
miss = routes[quality_cols].isna().sum()
print("Missing values (selected columns):")
print(pd.DataFrame({"missing": miss, "pct": (miss / len(routes) * 100).round(1)}))
print(f"\nUnmapped borough codes: {routes['boro_name'].isna().sum()}")
print(f"Segments with unparseable installation date: {routes['install_year'].isna().sum()}")

## 4. Station Metadata (GBFS)

Current station information and live availability from the Lyft GBFS feed (the same feed the
backend uses). Two endpoints are merged: `station_information` (static: name, location, capacity)
and `station_status` (live: bikes/docks available, operational flags).

In [ ]:
info = requests.get(GBFS_INFO_URL, timeout=(5, 30)).json()["data"]["stations"]
status = requests.get(GBFS_STATUS_URL, timeout=(5, 30)).json()["data"]["stations"]
status_map = {s["station_id"]: s for s in status}

rows = []
for s in info:
    st = status_map.get(s["station_id"], {})
    counts = {v.get("vehicle_type_id"): v.get("count", 0)
              for v in st.get("vehicle_types_available", [])}
    rows.append({
        "station_id": s["station_id"],
        "short_name": s.get("short_name"),
        "name": s.get("name"),
        "lat": s.get("lat"),
        "lon": s.get("lon"),
        "capacity": s.get("capacity"),
        "num_bikes_available": st.get("num_bikes_available"),
        "num_classic": counts.get("1"),
        "num_ebikes": counts.get("2"),
        "num_docks_available": st.get("num_docks_available"),
        "is_installed": st.get("is_installed"),
        "is_renting": st.get("is_renting"),
        "is_returning": st.get("is_returning"),
    })
stations = pd.DataFrame(rows)
stations["active"] = (
    (stations["is_installed"] == 1) & (stations["is_renting"] == 1) & (stations["is_returning"] == 1)
)
print(f"Stations in feed: {len(stations)}  (active: {stations['active'].sum()})")
stations.head()

### 4.1 Capacity and Live Availability

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(stations["capacity"].dropna(), bins=30, ax=axes[0]).set(
    title="Station capacity", xlabel="Docks")
avail = stations[["num_bikes_available", "num_docks_available"]].sum()
sns.barplot(x=avail.index, y=avail.values, ax=axes[1])
axes[1].set(title="Live system-wide availability", ylabel="Count")
plt.tight_layout()
plt.show()

print(f"Total bikes available now : {int(stations['num_bikes_available'].sum())}")
print(f"  of which e-bikes        : {int(stations['num_ebikes'].sum())}")
print(f"Total docks available now : {int(stations['num_docks_available'].sum())}")

### 4.2 Station Map

In [ ]:
station_map = folium.Map(location=[NYC_LAT, NYC_LON], zoom_start=12)
for _, s in stations.dropna(subset=["lat", "lon"]).iterrows():
    folium.CircleMarker(
        [s["lat"], s["lon"]],
        radius=2,
        color="green" if s["active"] else "gray",
        fill=True,
    ).add_to(station_map)
station_map

### 4.3 Data Quality

In [ ]:
print(f"Inactive stations (not installed/renting/returning): {(~stations['active']).sum()}")
print(f"Stations missing capacity : {stations['capacity'].isna().sum()}")
print(f"Stations missing coords   : {stations[['lat', 'lon']].isna().any(axis=1).sum()}")

# The numeric station_id (GBFS) differs from the public short_name used in the trip files;
# joins between live stations and historical trips must go through short_name.
print("\nExample station_id vs short_name:")
print(stations[["station_id", "short_name", "name"]].head())

The numeric `station_id` from GBFS is **not** the identifier used in the historical trip files —
those use the public `short_name`. Any join between live station metadata and trip data must go
through `short_name`. Inactive stations also remain in the feed and are filtered out for live views.